# 02 - Exploratory Data Analysis (EDA)
**Proyek:** Hotel Booking Cancellation Prediction | **Tahap:** M3 | **PRD:** v1.1

Alur: statistik deskriptif → distribusi target → distribusi fitur → hubungan fitur dengan target → korelasi → perbandingan set fitur untuk input aplikasi.
Setiap visualisasi diikuti **interpretasi**. Interpretasi di bawah ditulis dari hasil run saya; cocokkan dengan output Anda, dan tulis ulang dengan kata-kata sendiri sebelum dipakai di PPT.

In [ ]:
import os, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import HistGradientBoostingClassifier

pd.set_option('display.max_columns', 50); sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
os.makedirs('reports/figures', exist_ok=True)

cands = (glob.glob('/kaggle/input/**/hotel_bookings.csv', recursive=True) + glob.glob('./hotel_bookings.csv')
         + glob.glob('./data/raw/hotel_bookings.csv') + glob.glob('/mnt/user-data/uploads/hotel_bookings.csv'))
assert cands, 'hotel_bookings.csv tidak ditemukan'
raw = pd.read_csv(cands[0])

# Hapus duplikat (keputusan PRD 7.5), lalu fitur turunan
cols_dup = [c for c in raw.columns if c != 'reservation_status_date']
df = raw.drop_duplicates(subset=cols_dup).reset_index(drop=True)
df['nights'] = df.stays_in_weekend_nights + df.stays_in_week_nights
df['guests'] = df.adults + df.children.fillna(0) + df.babies
print('Data asli:', raw.shape, '| Setelah dedup:', df.shape)

## 1. Statistik deskriptif

In [ ]:
num_cols = ['lead_time','nights','guests','adr','total_of_special_requests','required_car_parking_spaces',
            'booking_changes','previous_cancellations']
df[num_cols].describe().T.round(2)

In [ ]:
print('Missing value (kolom terkait):')
print(df[['children','country','agent','company']].isna().sum())
print('Duplikat tersisa:', df.duplicated(subset=cols_dup).sum())

**Interpretasi:** `lead_time` rata-rata ≈ 80 hari tetapi median 49, dengan maksimum 737 hari, artinya distribusinya miring ke kanan (sedikit pemesanan yang sangat jauh menarik rata-rata ke atas). `adr` juga miring dengan maksimum 5.400, nilai ekstrem yang kemungkinan salah input. Duplikat sudah nol setelah pembersihan.

## 2. Distribusi target

In [ ]:
fig, ax = plt.subplots(figsize=(4.5,3.5))
sns.countplot(x='is_canceled', data=df, ax=ax)
for p in ax.patches: ax.annotate(int(p.get_height()), (p.get_x()+p.get_width()/2, p.get_height()), ha='center', va='bottom')
ax.set_title('Distribusi target (tanpa duplikat)'); plt.tight_layout()
plt.savefig('reports/figures/target_dist.png', dpi=120); plt.show()
print(df.is_canceled.value_counts(normalize=True).mul(100).round(2))

**Interpretasi:** dari 87.151 reservasi, sekitar 72,7% tidak dibatalkan dan 27,3% dibatalkan. Kelas tidak seimbang secara ringan. Konsekuensinya, accuracy bisa menyesatkan (model yang selalu menebak "tidak batal" sudah ≈ 72,7%), sehingga Recall, F1, dan PR-AUC kelas batal perlu diutamakan.

## 3. Distribusi fitur numerik dan outlier

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16,6))
for i, c in enumerate(['lead_time','nights','guests','adr']):
    sns.histplot(df[c], bins=50, ax=axes[0,i]); axes[0,i].set_title(f'Histogram {c}')
    sns.boxplot(x=df[c], ax=axes[1,i]); axes[1,i].set_title(f'Boxplot {c}')
plt.tight_layout(); plt.savefig('reports/figures/numeric_dist.png', dpi=120); plt.show()

rows = []
for c in ['lead_time','nights','guests','adr']:
    q1, q3 = df[c].quantile([.25,.75]); iqr = q3-q1
    n_out = ((df[c] < q1-1.5*iqr) | (df[c] > q3+1.5*iqr)).sum()
    rows.append({'fitur': c, 'median': df[c].median(), 'max': df[c].max(), 'jumlah_outlier_IQR': n_out})
pd.DataFrame(rows)

**Interpretasi:** keempat fitur miring ke kanan dan punya ekor panjang. Median `lead_time` 49 hari, `nights` 3, `guests` 2 (mayoritas pasangan/dua tamu), dan `adr` ≈ 98. Outlier IQR yang tercatat: `lead_time` 2.371, `nights` 3.008, `adr` 2.485. Untuk `guests`, angka outlier IQR (30.264) menyesatkan karena kebanyakan data bernilai 2 sehingga IQR sangat sempit; nilai 3-4 tamu tetap wajar. Yang benar-benar mencurigakan adalah nilai ekstrem seperti `adults` sampai 55, `adr` 5.400 atau negatif, dan `guests` = 0 (166 baris).
**Keputusan sementara (M4):** model berbasis pohon tidak sensitif terhadap outlier; untuk Logistic Regression gunakan scaling dan pertimbangkan batas atas (clipping) pada `lead_time`/`nights`. Baris dengan tamu 0 diputuskan di M4 beserta dampaknya.

## 4. Variabel kategorikal dan hubungannya dengan target

In [ ]:
def rate_plot(col, order=None, top=None, ax=None):
    t = df.groupby(col).is_canceled.agg(['count','mean']).rename(columns={'mean':'rasio_batal'})
    t = t.sort_values('count', ascending=False)
    if top: t = t.head(top)
    if order: t = t.reindex(order)
    sns.barplot(x=t.index.astype(str), y=t.rasio_batal, ax=ax); ax.set_ylabel('rasio batal')
    ax.set_title(f'Rasio pembatalan per {col}'); ax.tick_params(axis='x', rotation=30)
    return t

fig, axes = plt.subplots(2, 2, figsize=(13,8))
t1 = rate_plot('hotel', ax=axes[0,0]); t2 = rate_plot('deposit_type', ax=axes[0,1])
t3 = rate_plot('market_segment', ax=axes[1,0]); t4 = rate_plot('customer_type', ax=axes[1,1])
plt.tight_layout(); plt.savefig('reports/figures/cat_cancel_rate.png', dpi=120); plt.show()
for t in (t1,t2,t3,t4): display(t.round(3))

**Interpretasi:**
- **Hotel:** City Hotel (29,8%) lebih sering batal daripada Resort Hotel (23,3%).
- **Deposit:** `Non Refund` batal 94,6% (1.017 baris), sangat berbeda dari `No Deposit` (26,5%). Ini pola yang janggal secara bisnis (deposit tak bisa dikembalikan justru hampir selalu batal) sehingga diperlakukan sebagai keterbatasan/anomali data dan **bukan klaim sebab-akibat**. Jumlahnya kecil (≈1% data).
- **Market segment:** `Online TA` batal 35,2% (51.505 baris), jauh di atas `Direct` (14,7%), `Offline TA/TO` (14,6%), dan `Corporate` (11,9%).
- **Customer type:** `Transient` batal 30,0% sementara `Transient-Party` 14,4%, `Contract` 16,3%, dan `Group` 9,9%.
Segmen dan tipe pelanggan jelas membedakan risiko pembatalan, sehingga layak jadi kandidat input aplikasi.

## 5. Lead time, bulan, negara

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17,4.5))
bins = pd.cut(df.lead_time, [-1,7,30,90,180,365,800], labels=['0-7','8-30','31-90','91-180','181-365','>365'])
lt = df.groupby(bins, observed=True).is_canceled.agg(['count','mean'])
sns.barplot(x=lt.index.astype(str), y=lt['mean'], ax=axes[0]); axes[0].set_title('Rasio batal per rentang lead_time (hari)')

months = ['January','February','March','April','May','June','July','August','September','October','November','December']
mt = df.groupby('arrival_date_month').is_canceled.mean().reindex(months)
sns.barplot(x=mt.index.str[:3], y=mt.values, ax=axes[1]); axes[1].set_title('Rasio batal per bulan kedatangan')

top = df.country.value_counts().head(10).index
ct = df[df.country.isin(top)].groupby('country').is_canceled.mean().sort_values(ascending=False)
sns.barplot(x=ct.index, y=ct.values, ax=axes[2]); axes[2].set_title('Rasio batal 10 negara terbanyak')
for a in axes: a.set_ylabel('rasio batal')
plt.tight_layout(); plt.savefig('reports/figures/leadtime_month_country.png', dpi=120); plt.show()
display(lt.round(3))
print('Cakupan 10 negara teratas: %.1f%% data' % (df.country.isin(top).sum()/df.country.notna().sum()*100))

**Interpretasi:**
- **Lead time:** rasio batal naik seiring jarak pemesanan: 8,4% untuk 0-7 hari, 25,3% (8-30), 31,8% (31-90), 34,7% (91-180), 39,3% (181-365). Semakin jauh dari hari kedatangan, semakin besar peluang batal; ini fitur yang kuat dan intuitif.
- **Bulan:** pembatalan lebih tinggi pada musim ramai (Agustus 32,1%, Juli 31,7%) dan lebih rendah pada November (20,9%) dan Januari (21,9%). Pengaruhnya lebih lemah dibanding lead time.
- **Negara:** Portugal (PRT) 35,3%, Brasil 36,3%, dan Italia 34,8% lebih tinggi dibanding Inggris (18,9%), Prancis (19,5%), dan Jerman (19,5%). Sepuluh negara teratas mencakup ≈ 82% data, sehingga untuk dropdown aplikasi cukup memakai top-N negara dan "Lainnya". Perlu diingat ada 177 kode negara dan `country` kosong pada 451 baris.

## 6. Permintaan khusus, parking, dan temuan penting

In [ ]:
sr = df.groupby('total_of_special_requests').is_canceled.agg(['count','mean']).round(3)
pk = df.groupby('required_car_parking_spaces').is_canceled.agg(['count','mean']).round(3)
display(sr); display(pk)
fig, axes = plt.subplots(1, 2, figsize=(11,4))
sns.barplot(x=sr.index, y=sr['mean'], ax=axes[0]); axes[0].set_title('Rasio batal per jumlah permintaan khusus')
sns.barplot(x=pk.index, y=pk['mean'], ax=axes[1]); axes[1].set_title('Rasio batal per slot parkir diminta')
plt.tight_layout(); plt.savefig('reports/figures/requests_parking.png', dpi=120); plt.show()

**Interpretasi:**
- **Special requests:** semakin banyak permintaan khusus, semakin kecil rasio batal (0 permintaan 32,9% → 3 permintaan 17,1%). Masuk akal: tamu yang sudah menyampaikan kebutuhan cenderung berniat datang.
- **⚠️ Parking:** semua reservasi dengan `required_car_parking_spaces` ≥ 1 (7.313 baris) **tidak pernah dibatalkan (0%)**. Pemisahan sempurna seperti ini hampir pasti artefak pencatatan (slot parkir dicatat setelah tamu benar-benar datang), bukan pola perilaku. Fitur ini adalah **kandidat leakage**: jika dipakai, skor model tampak lebih bagus tetapi tidak jujur karena informasinya baru ada setelah hasilnya diketahui.
- **Keputusan:** `required_car_parking_spaces` **tidak dipakai** sebagai fitur maupun input aplikasi. Ini mengubah rekomendasi 5 input pada PRD v1.1 (lihat bagian 8). `total_of_special_requests` tetap dipakai dengan catatan di disclaimer bahwa nilainya bisa bertambah mendekati kedatangan.

## 7. Korelasi fitur numerik

In [ ]:
corr_cols = ['is_canceled','lead_time','nights','guests','adr','total_of_special_requests',
             'booking_changes','previous_cancellations','is_repeated_guest','days_in_waiting_list']
corr = df[corr_cols].corr()
plt.figure(figsize=(9,7)); sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Korelasi (Pearson) fitur numerik'); plt.tight_layout()
plt.savefig('reports/figures/corr_heatmap.png', dpi=120); plt.show()
corr['is_canceled'].drop('is_canceled').sort_values()

**Interpretasi:** korelasi linear tiap fitur dengan target semuanya lemah (|r| ≤ ≈ 0,18). Yang tertinggi: `lead_time` (+0,18), `adr` (+0,13), sedangkan `total_of_special_requests` (-0,12) dan `is_repeated_guest` (-0,09) berkorelasi negatif. Korelasi rendah tidak berarti fitur tidak berguna, karena hubungan sebenarnya bisa non-linear atau melalui interaksi (contoh: `market_segment` dan `country` yang bersifat kategorikal tidak tertangkap Pearson). Ini juga alasan membandingkan model non-linear (Random Forest, XGBoost) dengan Logistic Regression.

## 8. Perbandingan set fitur untuk input aplikasi
Agar tidak mengintip data uji, pemilihan fitur memakai **cross-validation 3-fold pada data training saja**. Split yang sama (80:20, stratified, `random_state=42`) akan dipakai kembali di M4. Model uji cepat: HistGradientBoosting, metrik ROC-AUC.

In [ ]:
cat_cols = ['hotel','arrival_date_month','deposit_type','market_segment','customer_type','country']
d2 = df.copy()
for c in cat_cols: d2[c] = d2[c].astype('category')
train, test = train_test_split(d2, test_size=0.2, stratify=d2.is_canceled, random_state=RANDOM_STATE)

def cv_auc(feats):
    m = HistGradientBoostingClassifier(categorical_features='from_dtype', max_iter=100, random_state=RANDOM_STATE)
    return cross_val_score(m, train[feats], train.is_canceled, cv=3, scoring='roc_auc', n_jobs=-1).mean()

sets = {
 'A. PRD v1.0 (6 input awal)': ['hotel','lead_time','arrival_date_month','nights','guests','deposit_type'],
 'B. lead_time + country + market_segment': ['lead_time','country','market_segment'],
 'C. B + special_requests': ['lead_time','country','market_segment','total_of_special_requests'],
 'D. C + customer_type (USULAN 5 input)': ['lead_time','country','market_segment','total_of_special_requests','customer_type'],
 'E. C + adr': ['lead_time','country','market_segment','total_of_special_requests','adr'],
 'F. C + parking (LEAKAGE, jangan dipakai)': ['lead_time','country','market_segment','total_of_special_requests','required_car_parking_spaces'],
}
res = pd.DataFrame({'set fitur': list(sets), 'n_fitur': [len(v) for v in sets.values()],
                    'CV ROC-AUC (train)': [round(cv_auc(v),3) for v in sets.values()]})
res

**Interpretasi:** set A (6 input PRD awal) paling lemah (≈0,70). Menambah `country`, `market_segment`, lalu `total_of_special_requests` menaikkan AUC secara bertahap hingga ≈0,83. Set F tampak paling tinggi (≈0,85) justru karena `parking` bocor, sehingga **tidak boleh dipilih**. Penggantinya yang jujur adalah `customer_type` (D ≈0,84) atau `adr` (E ≈0,84). Dipilih `customer_type` karena berupa dropdown 4 pilihan yang mudah diisi pengguna awam.

## 9. Kesimpulan M3
1. Target imbalance ringan (≈27% batal); utamakan Recall, F1, PR-AUC kelas batal.
2. Fitur paling berpengaruh secara visual: `lead_time`, `market_segment`, `country`, `customer_type`, `total_of_special_requests`.
3. **`required_car_parking_spaces` adalah leakage** (0% batal pada semua baris dengan parkir ≥ 1) dan dikeluarkan; `reservation_status*` juga dikeluarkan.
4. `deposit_type = Non Refund` (≈94,6% batal) adalah anomali data; dicatat sebagai keterbatasan.
5. **Usulan 5 input final:** `lead_time`, `country`, `market_segment`, `total_of_special_requests`, `customer_type`.
6. Outlier ekstrem (`adr`, `adults`, tamu = 0) diputuskan di M4.

**Langkah berikutnya: M4 Preprocessing** (`03_preprocessing.ipynb`): penanganan missing `country`, nilai tidak wajar, grouping negara jarang, encoding, scaling, split 80:20 stratified, dan `Pipeline`/`ColumnTransformer`.